# ELENDRIL V7 · Kaggle T4×2

**Owner-authorized automatic launch:** Run All starts the complete registered study after source, GPU, learning, recovery and capacity checks pass. `RUN_STUDY = True` is already set. Set it to False only for an explicit offline preview. Native Kaggle PyTorch is used; no CUDA stack is installed or replaced. GPU qualification remains pending an actual successful run.

Pinned source commit: `7e0bc1cf3f37b621f8be057159fe4ed22120a1af`. The campaign obtains its own data; no dataset input is required. Results and failures remain in `/kaggle/working/elendril-v7`. Both archive links appear below even when the command fails; a link is usable only when the corresponding ZIP exists.


In [ ]:
from pathlib import Path
SOURCE_SHA = "7e0bc1cf3f37b621f8be057159fe4ed22120a1af"
RUN_STUDY = True  # Owner-authorized Run All; False is an explicit offline preview.
OUTPUT = Path('/kaggle/working/elendril-v7')
assert type(RUN_STUDY) is bool


In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path
from uuid import uuid4
from IPython.display import FileLink, display

REPOSITORY = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
REPO = Path('/kaggle/working') / ('elendril-v7-source-' + SOURCE_SHA)
campaign_failed = False
if RUN_STUDY:
    try:
        if not re.fullmatch(r'[0-9a-f]{40}', SOURCE_SHA):
            raise ValueError('Invalid pinned source commit')
        # An interrupted fresh clone leaves its uniquely named staging directory
        # for inspection; a restart uses a new staging directory.
        if not REPO.exists():
            stage = REPO.parent / ('.elendril-v7-source-' + uuid4().hex)
            stage.mkdir(parents=True)
            subprocess.run(['git', 'init', '-q'], cwd=stage, check=True, timeout=30)
            subprocess.run(['git', 'remote', 'add', 'origin', REPOSITORY], cwd=stage, check=True, timeout=30)
            subprocess.run(['git', 'fetch', '--depth=1', 'origin', SOURCE_SHA], cwd=stage, check=True, timeout=240)
            subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=stage, check=True, timeout=60)
            if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=stage, text=True, timeout=30).strip() != SOURCE_SHA:
                raise RuntimeError('Fetched source does not match the pinned commit')
            stage.rename(REPO)
        if REPO.is_symlink() or not REPO.is_dir():
            raise RuntimeError('Pinned checkout must be a real directory')
        head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True, timeout=30).strip()
        dirty = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=all'], cwd=REPO, text=True, timeout=30).strip()
        if head != SOURCE_SHA or dirty:
            raise RuntimeError('Pinned checkout is not exact and clean; inspect it before retrying')
        env = os.environ.copy()
        env['PYTHONDONTWRITEBYTECODE'] = '1'
        subprocess.run([sys.executable, '-m', 'tools.elendril_v7_campaign',
                        '--output', str(OUTPUT), '--owner-enabled', '--source', SOURCE_SHA],
                       cwd=REPO, env=env, check=True)
        print('Campaign delivery completed: inspect FINAL_STATUS.json and scientific_report.md. Completion does not establish a root cause or scientific success.')
    except Exception as exc:
        campaign_failed = True
        print(f'Campaign did not complete: {type(exc).__name__}: {exc}')
        print('Existing raw and partial outputs remain available for inspection.')
else:
    print('Study disabled. No source download or campaign launched.')

# Links remain visible even after checkout, admission, or campaign failure.
for label, name in [('Analysis', 'ELENDRIL_V7_ANALYSIS.zip'), ('Full', 'ELENDRIL_V7_FULL.zip')]:
    path = OUTPUT / 'exports' / name
    print(f'{label} archive: {path} (exists: {path.is_file()})')
    display(FileLink(str(path)))
if RUN_STUDY:
    display(FileLink(str(OUTPUT / 'exports' / 'FINAL_STATUS.json')))
if campaign_failed:
    raise RuntimeError('Campaign failed or is incomplete. Inspect NO-GO/PARTIAL receipts and download available outputs; do not treat this as a successful study.')
